In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().resolve().parents[0]))

import json
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from utils.experiment_tracker import ExperimentTracker

In [ ]:
from training_pipeline.src.validation.validation_suite import ValidationSuite

validator = ValidationSuite()

In [ ]:
data_path = Path.cwd().resolve().parents[0] / "datasets" / "disaster_tweets.csv"

df = pd.read_csv(
    data_path,
    encoding="ISO-8859-1",
    engine="python",
    on_bad_lines="skip"
)

print("Dataset loaded successfully")
print(df.head())

validator.validate_dataset(df)

In [ ]:
df = df[df["choose_one"].isin(["Relevant", "Not Relevant"])].copy()
df = df.dropna(subset=["text"])

validator.validate_filtered_dataset(df)

X = df["text"]
y = df["choose_one"].map({
    "Relevant": 1,
    "Not Relevant": 0
})

print("Filtered dataset size:", len(df))
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Train/test split and vectorization completed")

In [ ]:
model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)

preds = model.predict(X_test_vec)

validator.validate_predictions(preds)

accuracy = accuracy_score(y_test, preds)
f1 = f1_score(y_test, preds)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

validator.validate_metrics(accuracy, f1)

In [ ]:
tracker = ExperimentTracker()

run_id = tracker.start_run(
    experiment_name="disaster_tweet_classifier",
    model_name="logistic_regression",
    model_version="lr_v1",
    dataset_name="disaster_tweets",
    dataset_version="disaster_tweets_v1",
    parameters={
        "model": "LogisticRegression",
        "max_iter": 200,
        "test_size": 0.2,
        "vectorizer": "TfidfVectorizer",
        "max_features": 5000,
        "stop_words": "english",
        "stratify": True
    }
)

tracker.log_metrics({
    "accuracy": float(accuracy),
    "f1_score": float(f1)
})

tracker.log_artifact("models/disaster_model.pkl")
tracker.end_run("completed")

print("Experiment tracking completed successfully")
print("Run ID:", run_id)

metadata = validator.validate_tracker_output(tracker.run_dir)
metadata

In [ ]:
print("W7-T4 validation test suite executed successfully")